# Sales Lead Assistant (RAG + Memory)
### Mid-term Project — Hagar Soliman

**Nova Digital Solutions** is a digital agency (websites, mobile apps, digital marketing, AI solutions).  
This assistant is added to their CRM to help the **sales team**: before calling any lead, the sales rep can ask for
a quick **brief** about the company and how to handle it.

- **Leads** → Google Sheet (200 companies)
- **Services, prices and sales rules** → PDF Sales Playbook
- **Memory** → remembers the whole conversation and understands follow-up questions

### Steps
1. Imports
2. Data Loading (PDF + Google Sheet)
3. Chunking
4. Embeddings + FAISS
5. Model Loading (Qwen)
6. Memory
7. Chat Function (RAG + Memory)
8. Testing
9. Evaluation
10. Summary Memory (Optional)
11. Chat With The Assistant

## 1. Imports

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf
!pip install -q bitsandbytes

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

/tmp/ipykernel_25143/3363962140.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. Data Loading
- **PDF** → `Nova_Digital_Sales_Playbook.pdf` (services, prices, sales rules) — upload it to `/content/`
- **Google Sheet** → `Nova_Digital_Leads` (200 leads) — shared as *Anyone with the link*

In [ ]:
pdf_path = "/content/Nova_Digital_Sales_Playbook.pdf"


sheet_id = "1PzyqWRR20jeSbgWrxTDG2x96rY7YJKbVjGXctUwirw4"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

### Load PDF

In [ ]:
pdf_pages = PyPDFLoader(pdf_path).load()

print("PDF pages:", len(pdf_pages))

PDF pages: 5


### Load Google Sheet
Every row becomes one sentence, like:  
`Company: Nile Pharma | Industry: Healthcare | Interested Service: Advanced App | ...`

In [ ]:
df = pd.read_csv(sheet_url)
df.head()

,Lead ID,Company,Industry,City,Employees,Interested Service,Budget (USD),Budget Type,Contact Person,Job Title,Stage,Lead Source,Last Contact,Notes
0,L001,Nile Pharma,Healthcare,Cairo,85,Advanced App,25000,One-time,Dr. Mona Adel,Operations Director,Proposal Sent,Facebook Ads,2026-10-12,Wants a mobile app for patients to book appoin...
1,L002,Delta Foods,Food & Beverage,Cairo,45,E-commerce Store,9000,One-time,Karim Fathy,Sales Manager,Contacted,Website,2026-10-09,Wants to sell products online with online paym...
2,L003,Cairo Logistics,Logistics,Alexandria,230,CRM + AI Integration,12000,One-time,Hany Saeed,CEO,Negotiation,Referral,2026-10-15,Needs to connect their SAP ERP with AI for aut...
3,L004,Pyramids Real Estate,Real Estate,Giza,18,SEO,500,Monthly,Salma Nour,Owner,New,Google Ads,2026-10-16,Website doesn't appear on Google search. Budge...
4,L005,Alex Textiles,Manufacturing,Giza,150,Business Automation,9100,One-time,Omar Khaled,IT Specialist,Contacted,Referral,2026-09-20,"Too much manual work in orders, invoices and r..."


In [ ]:
sheet_docs = []

for _, row in df.iterrows():
    row_text = (
        f"{row['Company']} ({row['Lead ID']}) is a {row['Industry']} company in {row['City']} with {row['Employees']} employees. "
        f"They are interested in: {row['Interested Service']}. "
        f"Budget: ${row['Budget (USD)']} ({row['Budget Type']}). "
        f"Contact person: {row['Contact Person']}, {row['Job Title']}. "
        f"Stage: {row['Stage']}. Lead source: {row['Lead Source']}. Last contact: {row['Last Contact']}. "
        f"Notes: {row['Notes']}"
    )
    sheet_docs.append(Document(page_content=row_text))

print("Sheet rows:", len(sheet_docs))
print(sheet_docs[0].page_content)

Sheet rows: 200
Nile Pharma (L001) is a Healthcare company in Cairo with 85 employees. They are interested in: Advanced App. Budget: $25000 (One-time). Contact person: Dr. Mona Adel, Operations Director. Stage: Proposal Sent. Lead source: Facebook Ads. Last contact: 2026-10-12. Notes: Wants a mobile app for patients to book appointments and pay online. Worried the price is high.


In [ ]:
print(df.shape)
print(len(sheet_docs[0].page_content))
print(sheet_docs[0].page_content[:300])

(200, 14)
365
Nile Pharma (L001) is a Healthcare company in Cairo with 85 employees. They are interested in: Advanced App. Budget: $25000 (One-time). Contact person: Dr. Mona Adel, Operations Director. Stage: Proposal Sent. Lead source: Facebook Ads. Last contact: 2026-10-12. Notes: Wants a mobile app for patient


## 3. Chunking
- **PDF** → split into small chunks
- **Sheet** → every row is already small, so each row = one chunk

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

pdf_chunks = text_splitter.split_documents(pdf_pages)

all_chunks = pdf_chunks + sheet_docs

print("PDF chunks  :", len(pdf_chunks))
print("Sheet chunks:", len(sheet_docs))
print("Total chunks:", len(all_chunks))

PDF chunks  : 29
Sheet chunks: 200
Total chunks: 229


## 4. Embeddings + FAISS

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(all_chunks, embeddings)
pdf_store = FAISS.from_documents(pdf_chunks, embeddings)

print("FAISS ready with", vector_store.index.ntotal, "vectors")

/tmp/ipykernel_25143/4285200260.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS ready with 229 vectors


### Semantic Search Test

In [ ]:
results = vector_store.similarity_search("How much does the Business Website cost?", k=3)

for doc in results:
    print(doc.page_content[:200])
    print("-" * 50)

Basic Website
$1,500
3 weeks
Up to 5 pages, mobile-friendly design, contact form, 1
year hosting
Business Website
$4,000
6 weeks
Up to 15 pages, content management system (CMS),
Arabic and English, ba
--------------------------------------------------
Basic Website, Business Website, E-commerce
Store
One-time project
price
Mobile App Development
Basic App, Standard App, Advanced App
One-time project
price
Digital Marketing
Social Media Management, 
--------------------------------------------------
4. Digital Marketing
 Package
Monthly
Price
Minimum
Contract
What is Included
Social Media
Management
$800 / month
3 months
3 platforms, 20 posts per month, monthly
report
Ads Management
$1,200 /
mont
--------------------------------------------------


## 5. Model Loading (Qwen)

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Model loaded on", device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded on cuda


### Generation Function

In [ ]:
def generate_response(messages, max_new_tokens=250):

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 6. Memory (Buffer Memory)
`conversation_history` saves **every** user question and assistant answer (full chat).

In [ ]:
conversation_history = []     # Buffer memory (full chat)
conversation_summary = ""     # used later in Summary Memory (optional)


def clear_memory():
    global conversation_summary
    conversation_history.clear()
    conversation_summary = ""
    print("Memory cleared.")


def show_memory():
    print("=" * 50)
    print("FULL CONVERSATION HISTORY")
    print("=" * 50)
    for i, turn in enumerate(conversation_history, 1):
        print(f"Turn {i}")
        print("  Sales Rep     :", turn["user"])
        print("  Assistant:", turn["assistant"])

## 7. Chat Function (RAG + Memory)

```
User question
     ↓
Search FAISS  (previous question + new question → handles follow-ups)
     ↓
Messages = Instructions + Context + Chat History + New Question
     ↓
Qwen → Answer
     ↓
Save the question and answer in memory
```

**Why add the previous question to the search?**  
If the user asks *"How many employees do they have?"*, the word *"they"* alone finds nothing.  
Adding the previous question (*"Tell me about Nile Pharma"*) makes the search find the right lead.

In [ ]:
SYSTEM_PROMPT = """You are a Sales Assistant inside the CRM of Nova Digital Solutions,
a digital agency that offers websites, mobile apps, digital marketing and AI solutions.
You help the sales team understand their leads before contacting them.

Rules:
1. Answer using ONLY the Context and the conversation history.
2. Never use your own general knowledge, even if you know the answer.
3. Use the conversation history to understand follow-up questions (words like it, they, them).
4. If the user asks about the conversation itself (previous questions, their name, which leads we discussed), answer from the conversation history.
5. If the user greets you or introduces themselves, reply politely.
6. Always answer in clear natural sentences. Never copy the raw data rows.
7. When comparing a budget with a price, compare the numbers carefully. Only suggest packages that cost less than or equal to the budget.
8. If the answer is not in the Context or the history, say: "I don't have this information."
9. Only mention prices, numbers and packages that appear in the Reference data. Never invent prices.
10. Keep answers short (2 to 4 sentences). Do not write emails or long lists unless the user asks for them."""

In [ ]:
def find_lead(text):
    # Returns the lead row if a company name is mentioned in the text
    for i, company in enumerate(df["Company"]):
        if company.lower() in text.lower():
            return sheet_docs[i]
    return None


def chat(question):

    # 1. Which lead are we talking about? (in this question, or the last one mentioned in the chat)
    lead = find_lead(question)
    if lead is None:
        for turn in reversed(conversation_history):
            lead = find_lead(turn["user"])
            if lead is not None:
                break

    # 2. Retrieve: the lead row + the most relevant rules from the PDF
    if conversation_history:
        search_query = conversation_history[-1]["user"] + " " + question
    else:
        search_query = question

    if lead is not None:
        docs = [lead] + pdf_store.similarity_search(lead.page_content, k=4) + pdf_store.similarity_search(search_query, k=4)
    else:
        docs = vector_store.similarity_search(question, k=4) + vector_store.similarity_search(search_query, k=4)

    context = "\n\n".join(doc.page_content for doc in docs)

    # 3. Build messages: instructions + (summary) + history + (context + new question)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if conversation_summary:
        messages.append({"role": "system", "content": "Summary of earlier conversation:\n" + conversation_summary})

    for turn in conversation_history:
        messages.append({"role": "user", "content": turn["user"]})
        messages.append({"role": "assistant", "content": turn["assistant"]})

    messages.append({
        "role": "user",
        "content": "Reference data from the knowledge base (this is NOT our conversation, do not copy it as is):\n"
                   + context
                   + "\n\nMy question: " + question
    })

    # 4. Generate the answer
    answer = generate_response(messages)

    # 5. Save in memory
    conversation_history.append({"user": question, "assistant": answer})

    return answer

In [ ]:
def ask(question):
    answer = chat(question)
    print("Sales Rep     :", question)
    print("Assistant:", answer)
    print("-" * 50)

## 8. Testing
### Test 1: Questions from the PDF

In [ ]:
clear_memory()

ask("What services does Nova Digital Solutions offer?")
ask("How much does the Business Website cost?")
ask("What is the maximum discount a sales rep can give without approval?")

Memory cleared.
Sales Rep     : What services does Nova Digital Solutions offer?
Assistant: Nova Digital Solutions offers websites, mobile apps, digital marketing, and AI solutions.
--------------------------------------------------
Sales Rep     : How much does the Business Website cost?
Assistant: The Business Website costs $4,000. It is delivered in 6 weeks and includes up to 15 pages, a content management system (CMS), support for Arabic and English, and basic SEO.
--------------------------------------------------
Sales Rep     : What is the maximum discount a sales rep can give without approval?
Assistant: A sales rep can give a discount of up to 10% without approval.
--------------------------------------------------


### Test 2: Questions from the Google Sheet

In [ ]:
clear_memory()

ask("Tell me about the lead Delta Foods.")
ask("Who is the contact person at Cairo Logistics?")

Memory cleared.
Sales Rep     : Tell me about the lead Delta Foods.
Assistant: Delta Foods is a Food & Beverage company in Cairo with 45 employees. They are interested in an E-commerce Store, with a budget of $9,000 (one-time). The contact person is Karim Fathy, the Sales Manager. They are currently in the Contacted stage, having been contacted on 2026-10-09. Their main goal is to sell products online with online payment, currently only done through distributors.
--------------------------------------------------
Sales Rep     : Who is the contact person at Cairo Logistics?
Assistant: The contact person at Cairo Logistics is Hany Saeed, the CEO.
--------------------------------------------------


### Test 3: Lead Brief + Follow-up Questions + Memory
The sales rep asks for a brief before a call, then asks follow-ups.  
The assistant must remember that **"they"** and **"it"** mean Nile Pharma.

In [ ]:
clear_memory()

ask("Give me a short brief about Nile Pharma before my call.")
ask("What service are they interested in?")
ask("Is their budget enough for it?")
ask("They said the price is too high. How should I respond?")
ask("Which company did I ask you about at the start?")

Memory cleared.
Sales Rep     : Give me a short brief about Nile Pharma before my call.
Assistant: Nile Pharma is a Healthcare company based in Cairo with 85 employees. The contact person is Dr. Mona Adel, the Operations Director. They are interested in developing a mobile app for patients to book appointments and pay online. Their budget is $25,000 for a one-time project, which matches the cost of the Advanced App package. Currently, the proposal has been sent, and the next action is to address their concern about the price being high.
--------------------------------------------------
Sales Rep     : What service are they interested in?
Assistant: Nile Pharma is interested in developing an Advanced App for patients to book appointments and pay online.
--------------------------------------------------
Sales Rep     : Is their budget enough for it?
Assistant: Their budget of $25,000 matches the cost of the Advanced App package, which includes a backend server and an admin dashboard. T

In [ ]:
show_memory()

FULL CONVERSATION HISTORY
Turn 1
  Sales Rep     : Give me a short brief about Nile Pharma before my call.
  Assistant: Nile Pharma is a Healthcare company based in Cairo with 85 employees. The contact person is Dr. Mona Adel, the Operations Director. They are interested in developing a mobile app for patients to book appointments and pay online. Their budget is $25,000 for a one-time project, which matches the cost of the Advanced App package. Currently, the proposal has been sent, and the next action is to address their concern about the price being high.
Turn 2
  Sales Rep     : What service are they interested in?
  Assistant: Nile Pharma is interested in developing an Advanced App for patients to book appointments and pay online.
Turn 3
  Sales Rep     : Is their budget enough for it?
  Assistant: Their budget of $25,000 matches the cost of the Advanced App package, which includes a backend server and an admin dashboard. This should cover their needs for a mobile app to allow pati

### Test 4: Question Outside the Data (should not make up an answer)

In [ ]:
clear_memory()

ask("Who won the World Cup in 2018?")

Memory cleared.
Sales Rep     : Who won the World Cup in 2018?
Assistant: I don't have this information.
--------------------------------------------------


## 9. Evaluation
We ask questions where we **know the correct answer**,  
then check if the correct answer appears in the assistant's answer.

In [ ]:
test_questions = [
    # From the PDF
    {"question": "How much does the Business Website cost?",                     "expected": "4000"},
    {"question": "How much does the E-commerce Store cost?",                     "expected": "8000"},
    {"question": "How many weeks does it take to deliver the Advanced App?",     "expected": "16"},
    {"question": "What is the monthly fee of the AI Chatbot?",                   "expected": "300"},
    {"question": "What discount do education organizations get?",               "expected": "20"},
    {"question": "What is the maximum discount a sales rep can give without approval?", "expected": "10"},
    {"question": "What discount does a lead get for buying 2 services together?", "expected": "15"},
    {"question": "How many months of free maintenance are included after delivery?", "expected": "3"},
    # From the Google Sheet
    {"question": "What industry is Nile Pharma in?",                             "expected": "healthcare"},
    {"question": "Who is the contact person at Delta Foods?",                    "expected": "karim"},
    {"question": "What stage is Cairo Logistics in?",                            "expected": "negotiation"},
    {"question": "What service is Fresh Market interested in?",                  "expected": "ads"},
    {"question": "What is the job title of the contact at Desert Farms?",        "expected": "intern"},
    {"question": "What service is MedLine Labs interested in?",                  "expected": "chatbot"},
]

In [ ]:
correct = 0

for test in test_questions:
    clear_memory()                      # each question is tested alone
    answer = chat(test["question"])

    is_correct = test["expected"] in answer.lower().replace(",", "")
    if is_correct:
        correct += 1

    print("✅" if is_correct else "❌", test["question"])
    print("   Answer:", answer[:150])

accuracy = correct / len(test_questions) * 100
print(f"\nAccuracy: {correct}/{len(test_questions)} = {accuracy:.0f}%")

Memory cleared.
✅ How much does the Business Website cost?
   Answer: The Business Website costs $4,000.
Memory cleared.
✅ How much does the E-commerce Store cost?
   Answer: The E-commerce Store costs $8,000.
Memory cleared.
✅ How many weeks does it take to deliver the Advanced App?
   Answer: The Advanced App takes 16 weeks to be delivered.
Memory cleared.
✅ What is the monthly fee of the AI Chatbot?
   Answer: The monthly fee for the AI Chatbot is $300 for hosting and updates, in addition to the initial setup cost of $5,000.
Memory cleared.
✅ What discount do education organizations get?
   Answer: Education organizations and non-profits get a 20% discount.
Memory cleared.
✅ What is the maximum discount a sales rep can give without approval?
   Answer: A sales rep can give a discount of up to 10% without approval.
Memory cleared.
✅ What discount does a lead get for buying 2 services together?
   Answer: A lead who buys 2 or more services together gets a 15% bundle discount.
Memory c

### Follow-up Evaluation (Memory)
The first question gives the topic, the second question is a follow-up.

In [ ]:
followup_tests = [
    {"first": "Tell me about the lead Nile Pharma.",  "followup": "How many employees do they have?", "expected": "85"},
    {"first": "Tell me about the lead Delta Foods.",  "followup": "What stage are they in?",         "expected": "contacted"},
    {"first": "Tell me about the lead MedLine Labs.", "followup": "Who is their contact person?",    "expected": "heba"},
    {"first": "What is the Business Website package?", "followup": "How many weeks does it take?",  "expected": "6"},
]

correct = 0

for test in followup_tests:
    clear_memory()
    chat(test["first"])
    answer = chat(test["followup"])

    is_correct = test["expected"] in answer.lower().replace(",", "")
    if is_correct:
        correct += 1

    print("✅" if is_correct else "❌", test["first"], "→", test["followup"])
    print("   Answer:", answer[:150])

print(f"\nFollow-up Accuracy: {correct}/{len(followup_tests)} = {correct / len(followup_tests) * 100:.0f}%")

Memory cleared.
✅ Tell me about the lead Nile Pharma. → How many employees do they have?
   Answer: Nile Pharma has 85 employees.
Memory cleared.
✅ Tell me about the lead Delta Foods. → What stage are they in?
   Answer: Delta Foods is in the Contacted stage. They were contacted on 2026-10-09 and are interested in an E-commerce Store with a budget of $9,000 (one-time).
Memory cleared.
✅ Tell me about the lead MedLine Labs. → Who is their contact person?
   Answer: Their contact person is Dr. Heba Mostafa, CEO.
Memory cleared.
✅ What is the Business Website package? → How many weeks does it take?
   Answer: The Business Website package is delivered in 6 weeks.

Follow-up Accuracy: 4/4 = 100%


## 10. Summary Memory (Optional)
For long chats, the full history becomes too long.  
**Solution:** when the history reaches 5 turns → summarize the oldest 3 and keep the last 2.

In [ ]:
SUMMARY_BATCH = 3
KEEP_RECENT = 2


def summarize_history():
    global conversation_summary, conversation_history

    if len(conversation_history) < SUMMARY_BATCH + KEEP_RECENT:
        return

    old_turns = conversation_history[:SUMMARY_BATCH]

    old_text = ""
    for turn in old_turns:
        old_text += f"User: {turn['user']}\nAssistant: {turn['assistant']}\n"

    messages = [
        {"role": "system", "content": "Summarize this conversation in short bullet points. Always keep: the user's name, the leads (companies) discussed, and the user's questions. Do not invent anything."},
        {"role": "user", "content": f"Old summary:\n{conversation_summary}\n\nNew conversation:\n{old_text}"}
    ]

    conversation_summary = generate_response(messages, max_new_tokens=250)
    conversation_history = conversation_history[SUMMARY_BATCH:]

    print("\n[Memory] Old turns summarized.\n")

In [ ]:
def chat_with_summary(question):
    answer = chat(question)      # same chat function (it already reads the summary)
    summarize_history()          # summarize if the history got long
    return answer

In [ ]:
clear_memory()

for q in [
    "Hi, my name is Hagar and I work in the sales team.",
    "Tell me about the lead Nile Pharma.",
    "What service are they interested in?",
    "What are the payment terms for projects?",
    "Tell me about the lead Elite Clinics.",
    "What is my name and which leads did we talk about?",
]:
    print("Sales Rep     :", q)
    print("Assistant:", chat_with_summary(q))
    print("-" * 50)

print("\nSUMMARY:\n", conversation_summary)

Memory cleared.
Sales Rep     : Hi, my name is Hagar and I work in the sales team.
Assistant: Hello Hagar! How can I assist you today with the leads?
--------------------------------------------------
Sales Rep     : Tell me about the lead Nile Pharma.
Assistant: Nile Pharma is a Healthcare company in Cairo with 85 employees. They are interested in an Advanced App for patients to book appointments and pay online. Their budget is $25,000 (one-time), and they are currently at the proposal sent stage. They are worried the price is high.
--------------------------------------------------
Sales Rep     : What service are they interested in?
Assistant: Nile Pharma is interested in an Advanced App for patients to book appointments and pay online.
--------------------------------------------------
Sales Rep     : What are the payment terms for projects?
Assistant: For projects like websites, mobile apps, and AI solutions, the payment terms are as follows: 50% when signing, 30% at the middle of

## 11. Chat With The Assistant
Type `memory` to see the history, `clear` to reset, `quit` to stop.

In [ ]:

clear_memory()

while True:
    question = input("You: ").strip()

    if question.lower() == "quit":
        break
    elif question.lower() == "memory":
        show_memory()
    elif question.lower() == "clear":
        clear_memory()
    elif question:
        print("Assistant:", chat(question), "\n")

Memory cleared.
You: Give me a short brief about MedLine Labs before my call.
Assistant: Company name: MedLine Labs  
Industry: Healthcare  
City: Giza  
Size: 200 employees  
Contact person: Dr. Heba Mostafa, CEO  
Service interest: AI Chatbot  
Budget: $6,000 (one-time)  
Current stage: Contacted  
Next action: Discuss the AI Chatbot solution  
Main problem: Receives hundreds of WhatsApp messages from patients asking about test results and prices.  
Best way to handle it: Offer a free 30-minute demo of an AI chatbot trained on their website content. 

You: They asked if the chatbot supports Arabic. What should I say?
Assistant: Yes, our AI Chatbots support Arabic, including the Egyptian dialect. They can be trained to answer in both Arabic and English during the setup process. This will ensure that MedLine Labs receives the best possible service and that their chatbot can effectively communicate with their patients in their preferred language. 

You: memory
FULL CONVERSATION HISTORY
